# Document Length as Evidence

**Computational Text Analysis Pack · Module 02 — Describing a Corpus**

*Technique:* Distributions of length; median vs mean; when a skew is the finding

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/02-describing-corpora/02d_document_length.ipynb)

Data: `masakhanews.csv`, `colonial_texts.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv', 'colonial_texts.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
df_colonial = pd.read_csv('colonial_texts.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

Length is the easiest thing to measure about a document and the one most often
skipped. It is worth attention for two reasons: it distorts other measures
(02c), and it is sometimes the finding.

It is also where the mean/median distinction bites hardest, because length
distributions are almost always skewed right — a few very long documents drag
the average away from anything typical.

## The tooling

**`collections.Counter`** is the whole engine of frequency analysis — a
standard-library dictionary that counts. **matplotlib** draws the distributions.
The field's dedicated tools (**AntConc**, **#LancsBox**) wrap the same counting
in a GUI.

In [ ]:
print(df["n_words"].describe().to_string())
print()
print(f"mean   : {df['n_words'].mean():>8,.0f} words")
print(f"median : {df['n_words'].median():>8,.0f} words")
print()
print("If those two differ substantially, the distribution is skewed and the")
print("median is the number to report. The mean is being pulled by outliers.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].hist(df["n_words"], bins=60)
axes[0].axvline(df["n_words"].mean(), color="crimson", label="mean")
axes[0].axvline(df["n_words"].median(), color="seagreen", label="median")
axes[0].set_xlabel("words per document"); axes[0].set_ylabel("documents")
axes[0].set_title("a long right tail"); axes[0].legend()

axes[1].hist(np.log10(df["n_words"].clip(lower=1)), bins=60)
axes[1].set_xlabel("log10(words per document)")
axes[1].set_title("the same data, log scale")
plt.tight_layout(); plt.show()

## Length by group

A boxplot compares whole distributions rather than single numbers, which is what
you want when the distributions are this skewed.

In [ ]:
GROUP = "category"          # ← try "language"

keep = df[GROUP].value_counts()
keep = keep[keep >= 30].index.tolist()
sub = df[df[GROUP].isin(keep)]

summary = (sub.groupby(GROUP)["n_words"]
             .agg(docs="size", median="median", mean="mean",
                  p10=lambda s: s.quantile(0.10), p90=lambda s: s.quantile(0.90))
             .sort_values("median", ascending=False))
print(summary.to_string(float_format=lambda v: f"{v:,.0f}"))

In [ ]:
order = summary.index.tolist()
fig, ax = plt.subplots(figsize=(9, 5))
ax.boxplot([sub[sub[GROUP] == g]["n_words"] for g in order],
           vert=False, showfliers=False)
ax.set_yticks(range(1, len(order) + 1))
ax.set_yticklabels(order)
ax.set_xlabel("words per document")
ax.set_title(f"Document length by {GROUP}")
plt.tight_layout(); plt.show()

### 🔧 Your turn 1

Switch `GROUP` to `"language"` and re-run.

Is the difference between languages about how those languages work, or about
which outlets the corpus sampled from? You cannot tell from this chart — and
saying so is the correct answer, not a failure.

## When length is the finding

In the colonial corpus every row is a fixed-size passage by construction, so
passage length says nothing. But **passages per work** does: it measures how long
each book is, which is a real property of the archive.

In [ ]:
per_work = (df_colonial.groupby(["work_id", "title", "author"])
                        .size().reset_index(name="passages")
                        .sort_values("passages", ascending=False))
print(f"{len(per_work)} works")
print()
print(per_work.head(8)[["title", "author", "passages"]].to_string(index=False))
print()
print("Note the ceiling: the corpus builder capped each work at 40 sampled")
print("passages. Anything at 40 is a book that hit the cap, not a book that is")
print("exactly that long. A limit imposed by the sampling is not a finding.")

### 🔧 Your turn 2

Count how many works sit exactly at the cap.

This is a general habit worth keeping: before interpreting the top of any
distribution, check whether it is a ceiling somebody imposed. Capped values,
rounded ages, and "999" codes all look like data.

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

**Your turn 1.** You cannot separate the two from this corpus. MasakhaNEWS drew
each language from different outlets, and outlets have house styles about article
length. A difference in median length between Hausa and Swahili reporting is
confounded with the publications sampled. The honest write-up says so and treats
length as a nuisance variable to control for, not a result.

**Your turn 2.** A substantial number sit exactly at 40 — those are the long
books, truncated. Any statistic computed over "passages per work" is therefore
censored from above, and the mean is meaningless. This is the text-analysis
version of the trap you met in the housing data with capped income bands.

</details>

## Where this stops

You can describe a corpus: how many documents, how long, how varied, what its
commonest words are. Everything so far is description of *one* corpus.

Next: **Module 03 — preparing text**, and the decisions that quietly change every
number you have computed.